# 03 · LSTM Autoencoder

Entrenamiento y evaluación del modelo bajo el flujo final: train para ajuste, holdout interno para early stopping, validation para umbral y test para evaluación final.

In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd

from scania_anomaly.config import load_config, ensure_directories
from scania_anomaly.utils.reproducibility import set_global_seed

config = load_config('../config/config.yaml')
ensure_directories(config)
set_global_seed(config['project']['seed'])

DRIVE_ROOT = Path(config['paths']['drive_root'])
RAW_DIR = Path(config['paths']['raw_dir'])
PROCESSED_DIR = Path(config['paths']['processed_dir'])
OUTPUTS_DIR = Path(config['paths']['outputs_dir'])
MODELS_DIR = Path(config['paths']['models_dir'])
METRICS_DIR = Path(config['paths']['metrics_dir'])
TABLES_DIR = Path(config['paths']['tables_dir'])
print(config['project']['name'], config['project']['version'])

import time
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split

from scania_anomaly.windowing import TimeWindowBuilder
from scania_anomaly.datasets import to_tensor_dataset
from scania_anomaly.training.trainer import TrainingConfig, AutoencoderTrainer
from scania_anomaly.outlier_detection import reconstruction_errors, select_threshold, classify_outliers
from scania_anomaly.vehicle_level import make_window_predictions, aggregate_vehicle_scores, classify_vehicle_scores
from scania_anomaly.model_evaluation import binary_classification_metrics
from scania_anomaly.experiment_tracking import save_json, save_predictions_table

from scania_anomaly.models.autoencoders import LSTMAutoencoder

def build_model(n_features: int):
    return LSTMAutoencoder(
        n_features=n_features,
        hidden_dim=config['modeling']['hidden_dim'],
        latent_dim=config['modeling']['latent_dim'],
    )


In [ ]:

train_data = TimeWindowBuilder.load_npz(PROCESSED_DIR / 'train_windows.npz')
validation_data = TimeWindowBuilder.load_npz(PROCESSED_DIR / 'validation_windows.npz')
test_data = TimeWindowBuilder.load_npz(PROCESSED_DIR / 'test_windows.npz')

train_ds = to_tensor_dataset(train_data)
val_holdout_size = max(1, int(len(train_ds) * config['modeling']['train_holdout_fraction']))
train_fit_size = max(1, len(train_ds) - val_holdout_size)
train_fit_ds, train_holdout_ds = random_split(train_ds, [train_fit_size, val_holdout_size], generator=torch.Generator().manual_seed(config['project']['seed']))

train_loader = DataLoader(train_fit_ds, batch_size=config['modeling']['batch_size'], shuffle=True)
holdout_loader = DataLoader(train_holdout_ds, batch_size=config['modeling']['batch_size'], shuffle=False)
validation_loader = DataLoader(to_tensor_dataset(validation_data), batch_size=config['modeling']['batch_size'], shuffle=False)
test_loader = DataLoader(to_tensor_dataset(test_data), batch_size=config['modeling']['batch_size'], shuffle=False)

n_features = train_data.X.shape[-1]
print('n_features:', n_features)


In [ ]:

model = build_model(n_features)
trainer = AutoencoderTrainer(model, TrainingConfig(
    epochs=config['modeling']['epochs'],
    learning_rate=config['modeling']['learning_rate'],
    batch_size=config['modeling']['batch_size'],
    patience=config['modeling']['early_stopping_patience'],
    device=config['modeling']['device'],
))

history = trainer.fit(train_loader, holdout_loader)
trainer.save(MODELS_DIR / 'lstm_autoencoder.pt')
save_json(history, METRICS_DIR / 'lstm_autoencoder_history.json')


In [ ]:

validation_scores = reconstruction_errors(model, validation_loader, trainer.device)
test_scores = reconstruction_errors(model, test_loader, trainer.device)

# Agregación a nivel vehículo para selección de umbral y evaluación principal.
val_window_df = make_window_predictions(validation_data.vehicle_ids, validation_scores, y_true=validation_data.y, start_time=validation_data.start_time, end_time=validation_data.end_time)
test_window_df = make_window_predictions(test_data.vehicle_ids, test_scores, y_true=test_data.y, start_time=test_data.start_time, end_time=test_data.end_time)

val_vehicle_df = aggregate_vehicle_scores(val_window_df)
test_vehicle_df = aggregate_vehicle_scores(test_window_df)

threshold = select_threshold(
    val_vehicle_df['max_score'].values,
    y_true=val_vehicle_df['y_true'].values,
    strategy=config['outlier_detection']['threshold_strategy'],
    percentile=config['outlier_detection']['threshold_percentile'],
)

val_vehicle_df = classify_vehicle_scores(val_vehicle_df, threshold, score_col='max_score')
test_vehicle_df = classify_vehicle_scores(test_vehicle_df, threshold, score_col='max_score')

metrics = binary_classification_metrics(test_vehicle_df['y_true'], test_vehicle_df['is_outlier'], scores=test_vehicle_df['max_score'])
metrics.update({'model': 'lstm_autoencoder', 'split': 'test', 'level': 'vehicle', 'threshold': threshold, **history})

save_json(metrics, METRICS_DIR / 'lstm_autoencoder_test_vehicle_metrics.json')
save_predictions_table(val_window_df, TABLES_DIR / 'lstm_autoencoder_validation_window_scores.csv')
save_predictions_table(test_window_df, TABLES_DIR / 'lstm_autoencoder_test_window_scores.csv')
save_predictions_table(val_vehicle_df, TABLES_DIR / 'lstm_autoencoder_validation_vehicle_scores.csv')
save_predictions_table(test_vehicle_df, TABLES_DIR / 'lstm_autoencoder_test_vehicle_scores.csv')
metrics
